# 01 · Data integrity and telemetry audit

Audits the **actual** `raw_results.csv`: shape/schema, seed/step coverage, missingness, ranges, and empirical distributions. The README seed list is compared rather than assumed correct.

In [ ]:
from pathlib import Path
import json, math, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def find_project():
    candidates=[Path.cwd(), Path.cwd()/"project_inputs", Path.cwd().parent/"project_inputs", Path("/mnt/data/diina_reproducibility/project_inputs"), Path("/mnt/data")]
    for p in candidates:
        if (p/"raw_results.csv").exists(): return p
    raise FileNotFoundError("Place notebook with project_inputs/ or set cwd to extracted project directory")
PROJECT=find_project()
df=pd.read_csv(PROJECT/"raw_results.csv")
summary=json.loads((PROJECT/"summary.json").read_text())
print("Project files:",PROJECT,"| telemetry shape:",df.shape,"| columns:",list(df.columns))
assert len(df)==1500 and set(df.columns)=={"seed","step","loss","fast_usage","slow_usage","entropy"}

In [ ]:
expected={42,123,456,789,2022}; observed=set(df.seed.unique())
print("Seeds:",sorted(observed)," expected per task:",sorted(expected))
print("Missing values:\n",df.isna().sum())
print("Rows per seed:\n",df.groupby("seed").size())
print("Steps per seed (min,max,unique):\n",df.groupby("seed").step.agg(["min","max","nunique"]))
assert df.groupby("seed").size().eq(300).all()
assert df.groupby("seed").step.apply(lambda x:set(x)==set(range(1,301))).all()
assert not df.isna().any().any()
assert df.loss.ge(0).all() and df.fast_usage.between(0,1).all() and df.slow_usage.between(0,1).all()
assert np.allclose(df.fast_usage+df.slow_usage,1,atol=1e-5)
print("PASS: 1,500 records; five seeds × 300 steps; usage sums to one.")

In [ ]:
display(df.describe().T)
fig,ax=plt.subplots(1,2,figsize=(11,3.6))
ax[0].hist(df.loss,bins=35,color="#34699a"); ax[0].set(title="Loss distribution",xlabel="loss",ylabel="step records")
ax[1].boxplot([g.entropy.values for _,g in df.groupby("seed")],tick_labels=[str(s) for s in sorted(df.seed.unique())]); ax[1].set(title="Entropy by seed",xlabel="seed")
plt.tight_layout()

**Audit note:** The CSV seeds are `42, 123, 789, 1011, 2022`. README instead says `42, 123, 789, 1011, 2022`? Compare the actual files in-cell; the requested validation seed list also differs. Logs, not prose, determine observed data.